In [3]:
import sys
sys.path.append("../")
import utils

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import WhiteKernel, RBF, ConstantKernel
from scipy import stats
import warnings
warnings.simplefilter('ignore')

# Parameters (change as needed)
csv_path = "../data/dataset_1.csv"   # adjust path if needed
seed = 0
n_init = 10
n_steps = 50
shuffle_seed = 0

# Load data
df_ = pd.read_csv(csv_path)

# Use last column as target (prop)
prop = list(df_.columns)[-1]

# Standardize explanatory features (all columns except the last)
df_.iloc[:, :-1] = StandardScaler().fit_transform(df_.iloc[:, :-1])

# Split target and features
df_oracle = df_[[prop]]
df_desc = df_.drop(prop, axis=1)

# Initial labeled sample
idx_sample = list(df_oracle.sample(n_init, random_state=seed).index)
df = df_oracle.iloc[idx_sample].reset_index(drop=True)
df_X = df_desc.iloc[idx_sample].reset_index(drop=True)

# Pool (remaining)
df_pool = df_oracle.drop(idx_sample).reset_index(drop=True)
df_pool_X = df_desc.drop(idx_sample).reset_index(drop=True)

# Shuffle pool
idx = list(df_pool_X.sample(frac=1, random_state=shuffle_seed).index)
df_pool = df_pool.iloc[idx].reset_index(drop=True)
df_pool_X = df_pool_X.iloc[idx].reset_index(drop=True)

# Records
obs_max = [df[prop].max()]
ec = []

for num in range(n_steps):
    # alpha schedule
    alpha = 1.0 / (num + 1) ** 0.5

    # Fit Gaussian process on current labeled set
    gp = GaussianProcessRegressor(kernel=ConstantKernel() * RBF() + WhiteKernel(), alpha=0, normalize_y=True)
    gp.fit(df_X.values, df[prop].values.ravel())

    # Predict on pool (mean and std)
    m, s = gp.predict(df_pool_X.values, return_std=True)

    # predictive interval bounds (two-sided, level = 1-alpha)
    y_pred_ub = stats.norm.ppf(1 - alpha * 0.5, loc=m, scale=s)
    y_pred_lb = stats.norm.ppf(alpha * 0.5, loc=m, scale=s)

    # Coverage on the pool
    ec.append(np.sum((df_pool[prop].values >= y_pred_lb) & (df_pool[prop].values <= y_pred_ub)) / len(df_pool))

    # Acquisition: expected improvement (integral) per pool point
    current_max = df[prop].max()
    acquisition_function = np.array([utils.EI_integral(m[i], s[i], current_max) for i in range(len(m))])

    # Select top-1 by acquisition (position index)
    new_pos = int(np.argmax(acquisition_function))

    # Move selected sample from pool to labeled set using positional index robustly
    df_X = pd.concat([df_X, df_pool_X.iloc[[new_pos]]]).reset_index(drop=True)
    df_pool_X = df_pool_X.drop(df_pool_X.index[new_pos]).reset_index(drop=True)

    df = pd.concat([df, df_pool.iloc[[new_pos]]]).reset_index(drop=True)
    df_pool = df_pool.drop(df_pool.index[new_pos]).reset_index(drop=True)

    # Record observed max
    obs_max.append(df[prop].max())